In [1]:
# train_with_frozen_stats.py
import os, json, time, numpy as np, torch
from pathlib import Path
from torch.utils.data import DataLoader
from dataset import UnifiedDesignDataset, design_collate_fn, split_designs
from models.film_model_v2 import FiLMNet
import time
import csv

# ----------------------- Paths -----------------------
CSV = "csv_files/case_with_geom_params_train.csv"
H5  = "data/surface_data.hdf5"
NORM_JSON = "norm_stats.json"   # produced from train-only stats earlier
CKPT_DIR  = Path("./checkpoints"); CKPT_DIR.mkdir(parents=True, exist_ok=True)
BEST_PATH  = CKPT_DIR / "film_best.pth"
FINAL_PATH = CKPT_DIR / "film_final.pth"
CFG_PATH   = CKPT_DIR / "train_config.json"

# ----------------------- Hyperparams -----------------------
EPOCHS =  1000
BATCH_SIZE = 32
LR = 10e-4
TRAIN_RATIO = 0.9

# ----------------------- Device & AMP -----------------------
cuda_avail = torch.cuda.is_available()
device = torch.device("cuda:0" if cuda_avail else "cpu")
if cuda_avail:
    print(f"✅ CUDA is available. Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ CUDA not available. Using CPU.")

use_amp = cuda_avail  # flip to False if you don't want AMP
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

# ----------------------- Load norm stats -----------------------
#raw = json.load(open(NORM_JSON))
#NORM = {k: np.array(v, dtype=np.float32) for k, v in raw.items()}

# ----------------------- Dataset & Loaders -----------------------
ds_full = UnifiedDesignDataset(csv_path=CSV, hdf5_path=H5, norm_stats=None, mode="train")
train_ds, val_ds = split_designs(ds_full, train_ratio=TRAIN_RATIO)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=design_collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=design_collate_fn)

# ----------------------- Model -----------------------
model = FiLMNet(cond_dim=13, coord_dim=6, output_dim=3,
                hidden_dim=256, num_layers=4, extra_layers=3).to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR)

# ----------------------- Save config -----------------------
cfg = dict(
    csv=str(CSV), h5=str(H5), norm_json=str(NORM_JSON),
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, train_ratio=TRAIN_RATIO,
    device=str(device), amp=use_amp, model=dict(cond_dim=13, coord_dim=6, output_dim=3,
                                                hidden_dim=256, num_layers=4, extra_layers=3)
)
json.dump(cfg, open(CFG_PATH, "w"), indent=2)

# ----------------------- Train/Val Loop -----------------------
best_val = float("inf")
t0 = time.time()
timelist = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    count = 0
    t1 = time.time()
    for coords, conds, targets in train_loader:
        coords, conds, targets = coords.to(device), conds.to(device), targets.to(device)

        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            pred = model(coords, conds)
            loss = torch.mean((pred - targets) ** 2)

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        running += loss.item() * coords.size(0)
        count   += coords.size(0)

    train_mse = running / max(count, 1)

    # ---- validation ----
    model.eval()
    v_running, v_count = 0.0, 0
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=use_amp):
        for coords, conds, targets in val_loader:
            coords, conds, targets = coords.to(device), conds.to(device), targets.to(device)
            pred = model(coords, conds)
            vloss = torch.mean((pred - targets) ** 2)
            v_running += vloss.item() * coords.size(0)
            v_count   += coords.size(0)
    val_mse = v_running / max(v_count, 1)

    # Save best
    if val_mse < best_val:
        best_val = val_mse
        torch.save(model.state_dict(), BEST_PATH)
        print(f"[epoch {epoch:03d}] train_mse={train_mse:.4e} | val_mse={val_mse:.4e}  <-- saved BEST")
    else:
        print(f"[epoch {epoch:03d}] train_mse={train_mse:.4e} | val_mse={val_mse:.4e}")
    timetaken = time.time() - t1
    timelist.append(timetaken)
    print(f"done in:{timetaken}")        

with open('time.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    for times in timelist:
        writer.writerow([times])

# ----------------------- Save final -----------------------
torch.save(model.state_dict(), FINAL_PATH)
dt = time.time() - t0
print(f"\nDone in {dt/60:.1f} min")
print(f"Best weights:  {BEST_PATH}")
print(f"Final weights: {FINAL_PATH}")


✅ CUDA is available. Using GPU: NVIDIA GeForce RTX 4070 Laptop GPU


C:\Users\samlu\AppData\Local\Temp\ipykernel_12908\2643928635.py:34: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


[dataset_v4] designs kept: 999, skipped rows: 3


C:\Users\samlu\AppData\Local\Temp\ipykernel_12908\2643928635.py:74: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
C:\Users\samlu\AppData\Local\Temp\ipykernel_12908\2643928635.py:90: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=use_amp):


[epoch 001] train_mse=1.0526e+00 | val_mse=5.9470e-01  <-- saved BEST
done in:15.531700372695923
[epoch 002] train_mse=6.9455e-01 | val_mse=5.9718e-01
done in:13.545393705368042
[epoch 003] train_mse=5.8026e-01 | val_mse=5.0597e-01  <-- saved BEST
done in:13.397009134292603
[epoch 004] train_mse=5.9890e-01 | val_mse=4.9933e-01  <-- saved BEST
done in:13.173084735870361
[epoch 005] train_mse=6.0454e-01 | val_mse=4.2483e-01  <-- saved BEST
done in:12.90272307395935
[epoch 006] train_mse=5.1054e-01 | val_mse=4.2576e-01
done in:12.994854927062988
[epoch 007] train_mse=3.8950e-01 | val_mse=5.2644e-01
done in:12.564019203186035
[epoch 008] train_mse=4.0346e-01 | val_mse=3.7904e-01  <-- saved BEST
done in:13.235530853271484
[epoch 009] train_mse=3.3989e-01 | val_mse=3.7502e-01  <-- saved BEST
done in:13.49893593788147
[epoch 010] train_mse=4.2878e-01 | val_mse=2.7836e-01  <-- saved BEST
done in:12.935481309890747
[epoch 011] train_mse=3.9919e-01 | val_mse=2.9271e-01
done in:12.567861795425415

In [2]:
import time
dt = time.time() - t0
print(f"\nDone in {dt/60:.1f} min")
print(f"Best weights:  {BEST_PATH}")
print(f"Final weights: {FINAL_PATH}")


Done in 133.1 min
Best weights:  checkpoints\film_best.pth
Final weights: checkpoints\film_final.pth
